In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Change the directory to your new Drive folder before running the rest of the code
os.chdir('/content/drive/MyDrive/wastewater-cdr-optimizer')

Mounted at /content/drive


In [2]:
!pip install -q geopandas shapely

import os
import pandas as pd

# 1. Create the repository folder structure
os.makedirs('data', exist_ok=True)
os.makedirs('src', exist_ok=True)

# 2. Generate mock EPA Wastewater Data (California coordinates)
epa_mock = pd.DataFrame({
    'facility_name': ['San Jose WWTP', 'Hyperion LA', 'Sacramento Regional'],
    'flow_mgd': [110, 260, 130],
    'Latitude': [37.42, 33.92, 38.45],
    'Longitude': [-121.95, -118.43, -121.49]
})
epa_mock.to_csv('data/california_epa_npdes_flows.csv', index=False)

# 3. Generate mock USGS Limestone Data
usgs_mock = pd.DataFrame({
    'site_name': ['Permanente Quarry', 'Lucerne Valley Quarry', 'Placerville Quarry'],
    'Latitude': [37.31, 34.35, 38.72],
    'Longitude': [-122.09, -116.89, -120.80]
})
usgs_mock.to_csv('data/california_usgs_mrds_limestone.csv', index=False)

print("Directories and mock data created.")

Directories and mock data created.


In [3]:
%%writefile src/spatial_ops.py
import pandas as pd
import geopandas as gpd
from typing import Tuple

class WAESpatialOptimizer:
    def __init__(self, epsg_code: int = 3310):
        """Initializes pipeline with standard California Albers projection."""
        self.epsg_code = epsg_code
        self.co2_yield_per_mgd = 60

    def load_and_project(self, epa_path: str, usgs_path: str) -> Tuple[gpd.GeoDataFrame, gpd.GeoDataFrame]:
        """Loads CSVs and projects to local CRS."""
        epa_df = pd.read_csv(epa_path)
        usgs_df = pd.read_csv(usgs_path)

        epa_gdf = gpd.GeoDataFrame(
            epa_df, geometry=gpd.points_from_xy(epa_df.Longitude, epa_df.Latitude), crs="EPSG:4326"
        )
        usgs_gdf = gpd.GeoDataFrame(
            usgs_df, geometry=gpd.points_from_xy(usgs_df.Longitude, usgs_df.Latitude), crs="EPSG:4326"
        )
        return epa_gdf.to_crs(epsg=self.epsg_code), usgs_gdf.to_crs(epsg=self.epsg_code)

    def compute_logistics_join(self, epa_gdf: gpd.GeoDataFrame, usgs_gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
        """Executes vectorized nearest-neighbor spatial join."""
        joined = gpd.sjoin_nearest(epa_gdf, usgs_gdf, how="left", distance_col="haul_distance_meters")
        joined['haul_distance_miles'] = joined['haul_distance_meters'] * 0.000621371
        return joined

    def calculate_viability(self, gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
        """Calculates yields and Viability Index."""
        gdf['est_co2_t_yr'] = gdf['flow_mgd'] * self.co2_yield_per_mgd
        gdf['viability_index'] = gdf['flow_mgd'] / (gdf['haul_distance_miles'] + 0.1)
        return gdf.sort_values(by='viability_index', ascending=False)

    def export_pipeline(self, epa_path: str, usgs_path: str, out_path: str):
        """Orchestrates pipeline and exports GeoJSON."""
        epa_gdf, usgs_gdf = self.load_and_project(epa_path, usgs_path)
        logistics_gdf = self.compute_logistics_join(epa_gdf, usgs_gdf)
        final_gdf = self.calculate_viability(logistics_gdf)

        final_gdf.to_crs(epsg=4326).to_file(out_path, driver="GeoJSON")
        print(f"Pipeline executed. Artifact saved to {out_path}")

Writing src/spatial_ops.py


In [4]:
# 1. Setup pathing to import from the src directory
import sys
import os
sys.path.append(os.path.abspath('src'))
from spatial_ops import WAESpatialOptimizer

# 2. Initialize and run pipeline
optimizer = WAESpatialOptimizer()
optimizer.export_pipeline(
    epa_path='data/california_epa_npdes_flows.csv',
    usgs_path='data/california_usgs_mrds_limestone.csv',
    out_path='ca_wwtp_cdr_viability.geojson'
)

# 3. Verify the output
import geopandas as gpd
results = gpd.read_file('ca_wwtp_cdr_viability.geojson')
display(results[['facility_name', 'flow_mgd', 'haul_distance_miles', 'viability_index']].head())

Pipeline executed. Artifact saved to ca_wwtp_cdr_viability.geojson


,facility_name,flow_mgd,haul_distance_miles,viability_index
0,San Jose WWTP,110,10.813170,10.079565
1,Sacramento Regional,130,41.707538,3.109487
2,Hyperion LA,260,93.096535,2.789803
